# (1) Data Description

**Files:**  
- `players.csv` → A list of all unique players, including data about each player. 
- `sessions.csv` → A list of individual play sessions by each player, including data about the session.


**What’s inside :**

**players.csv**  
- `hashedEmail` (text)– player ID  
- `subscribe` (TRUE/FALSE) – newsletter sign-up  
- `Age` (number) – age in years  
- `gender` (text) – self-reported  
- `name` (text) – first name  
- `experience` (text) – level label (e.g., Amateur, Pro)  
- `played_hours` (number) – total hours on record (player-level)

**sessions.csv**  
- `hashedEmail` (text) – join key  
- `start_time`, `end_time` (text) – `DD/MM/YYYY HH:MM`  
- `original_start_time`, `original_end_time` (number) – timestamps

**What we will make from sessions:**  
- `session_minutes` = end − start (minutes)  
- Per player: `total_playtime_min`, `avg_session_min`, `session_count`

**one-number summaries:**  
- We report the **mean** (2 decimal places) for each **number** in `players.csv`.

**Possible issues:**  
- Missing values in times or age; bad timestamps → NA durations.  
- Players with no sessions (messes up the engagement features).  
- Units differ: `played_hours` (hours) vs `total_playtime_min` (minutes).  

**Collected data:**  
- The server logged sessions automatically (start/end).  
- Player info and newsletter choice likely from sign-up forms.  
- Linked by `hashedEmail`.

---

# (2) Questions

**Broad question (Q1):**  
What player behaviours are most predictive of subscribing to the newsletter?

**Specific question:**  
Can player engagment behaviours like **total playtime**, **average session length**, and **session frequency** predict whether a player **subscribes** to the newsletter?

**How the data helps:**  
- `players.csv` has the **subscribe** flag.  
- `sessions.csv` tells me **how much** and **how often** each person plays.  
- Join on `hashedEmail`, build the three engagement features, then compare engagement vs subscribe.

**Wrangling plan:**  
1) convert time text to datetimes → minutes: `session_minutes = end - start` (droping the bad rows).  
2) Group by `hashedEmail` → make `total_playtime_min`, `avg_session_min`, `session_count`.  
3) join these to `players.csv` on `hashedEmail`.  
4) Fix units/types and keep only needed columns for plots/model.


---

 # (3) Exploratory Data Analysis and Visualization
We will load both datasets (`players.csv` and `sessions.csv`) into R.  
Then we will do some cleaning — convert the session start and end times into minutes, average them per player, and join that with player info.

Next, we will calculate the **mean** for all numeric columns in `players.csv` (to 2 decimals) and put them in a table.  
Finally, we will make two simple plots:
1) A **histogram** to see how total playtime is spread.  
2) A **boxplot** to compare playtime for players who subscribed vs those who didn’t.

**Insights:**  
- Players who subscribed seem to have **more total playtime**.  
- A few players play way more than others (outliers).  
- Data looks fine overall but has some missing and very long session values that I’ll fix later.

In [ ]:
library(tidyverse)
library(lubridate)

players <- read_csv("players.csv", show_col_types = FALSE)
sessions <- read_csv("sessions.csv", show_col_types = FALSE)

sessions_clean <- sessions |>
  mutate(start = dmy_hm(start_time, quiet = TRUE),
    end = dmy_hm(end_time, quiet = TRUE),
    session_minutes = as.numeric(difftime(end, start, units = "mins"))) |>
  filter(!is.na(session_minutes), session_minutes >= 0)


player_sessions <- sessions_clean |>
  group_by(hashedEmail) |>
  summarise( total_playtime_min = sum(session_minutes, na.rm = TRUE),
    avg_session_min = mean(session_minutes, na.rm = TRUE),
    session_count = n(),
    .groups = "drop")


data <- players |>
  left_join(player_sessions, by = "hashedEmail") |>
  mutate(played_minutes = played_hours * 60)

players_means <- players |>
  select(where(is.numeric)) |>
  summarise(across(everything(), ~ round(mean(.x, na.rm = TRUE), 2))) |>
  pivot_longer(everything(), names_to = "Variable", values_to = "Mean (2dp)")
print(players_means)

# Histogram
ggplot(data, aes(x = total_playtime_min)) +
  geom_histogram(bins = 30, fill = "skyblue", color = "black") +
  labs(title = "Distribution of Total Playtime", x = "Total Playtime (minutes)", y = "Number of Players") +
  theme_minimal()

# Boxplot
ggplot(data, aes(x = as.factor(subscribe), y = total_playtime_min, fill = as.factor(subscribe))) +
  geom_boxplot() +
  labs(title = "Total Playtime by Subscription", x = "Subscribed (FALSE/TRUE)", y = "Total Playtime (minutes)") +
  theme_minimal()


# (4) Methods and Plan

**Method:** Logistic Regression  

**Why:**  
Our response variable (`subscribe`) is binary (TRUE/FALSE), so a logistic regression is the best way to predict it. It also shows how each player behavior (like total playtime or session count) affects the chance of subscribing.

**Assumptions:**  
- Each player is independent.  
- The predictors affect the result in a steady, not random, way.
- No extreme outliers or strong correlations between predictors.  

**Limitations:**  
- It can’t capture complex or nonlinear patterns.  
- Sensitive to missing data and outliers.  
- Results just show correlation, nothing else. 

**How we will compare models:**  
We will start with one logistic regression and may compare it with a simple model to check performance. We can use metrics like accuracy and percision.

**Data process:**  
We will split the dataset into **70% training** and **30% testing**, using a **5-fold cross-validation** on the training set to tune and check stability before testing the final model. 
Before fitting the model, we will check for a class imbalance between subscribers and non-subscribers. If one group is larger than the other, the accuracy may be misleading, so extra metrics like precision and recall will have to be used. We will also use the categorical variables (gender and experience) in the model. In addition, We will look for multicollinearity.